Cell 1 — Install system deps + Ollama


In [2]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,496 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently

Cell 2 — Start Ollama server


In [3]:
import subprocess, time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(8)
print("Ollama server started")

Ollama server started


Cell 3 — Pull MKLLM + create Modelfile + register ← changed


In [4]:
# Pull GGUF (~7.7GB)
!ollama pull hf.co/NikolayKozloff/MKLLM-7B-Instruct-Q8_0-GGUF:Q8_0

modelfile_content = """FROM hf.co/NikolayKozloff/MKLLM-7B-Instruct-Q8_0-GGUF:Q8_0

PARAMETER temperature 0.1
PARAMETER repeat_penalty 1.3
PARAMETER num_predict 800
PARAMETER num_ctx 4096

TEMPLATE \"\"\"{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
\"\"\"

SYSTEM \"Разговор помеѓу љубопитен корисник и асистент со вештачка интелигенција. Асистентот дава корисни, детални и љубезни одговори на прашањата на корисникот.\""""

with open("Modelfile_MKLLM", "w") as f:
    f.write(modelfile_content)

!ollama create mkllm-7b-instruct -f Modelfile_MKLLM
print("✅ mkllm-7b-instruct registered")



✅ mkllm-7b-instruct registered


Cell 4 — Smoke test


In [5]:
import requests

def test_ollama_model(model_name: str, prompt: str = "Кажи ми нешто на македонски."):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model_name, "prompt": prompt, "stream": False},
            timeout=300
        )
        if response.status_code == 200:
            result = response.json()
            print("✅ Model is working!")
            print(f"Prompt: {prompt}")
            print(f"Response: {result['response']}")
            return True
        else:
            print(f"❌ Model returned status {response.status_code}: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error testing model: {e}")
        return False

test_ollama_model("mkllm-7b-instruct")

✅ Model is working!
Prompt: Кажи ми нешто на македонски.
Response: Здраво! Как можете? Дали имам можност да ви помогнам со било какви проблеми што ги имате денес или воопшто? Ако сакате, слободно прашајте.


True

Cell 5,6 — Install spacy + models + stanza


In [6]:
!pip install spacy -q
!python -m spacy download xx_ent_wiki_sm -q
!python -m spacy download mk_core_news_lg
!pip install stanza -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 82.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.8/325.8 MB 5.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('mk_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 54.0 MB/s eta 0:00:00
   ━━━━

Cell 7 — Install pip packages


In [7]:
!pip install langchain langchain-ollama langchain-community itext2kg pymupdf neo4j sentence-transformers -q
!pip install langchain-core==0.3.85 -q --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.

Cell 8 — Verify imports


In [1]:
import langchain_core
print(langchain_core.__version__)  # should be 0.3.85

from itext2kg import iText2KG_Star, DocumentsDistiller
from langchain_ollama import ChatOllama
print("✅ all imports ok")

0.3.85
✅ all imports ok


Cell 9 — All imports


In [2]:
import requests
from itext2kg import iText2KG_Star, DocumentsDistiller
import gc
import yaml
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, field_validator
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from difflib import SequenceMatcher
import os
import re
import asyncio
from typing import List, Optional
from itext2kg.logging_config import get_logger
from langchain_core.caches import BaseCache
import fitz
from neo4j import GraphDatabase
from langchain_community.embeddings import HuggingFaceEmbeddings
import spacy
import time
import gc
import types
from typing import List

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")
ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()
ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()

Cell 10 — Restart Ollama server


In [3]:
import subprocess, time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(8)
print("Ollama server started")

Ollama server started


Cell 11 — Test embeddings


In [4]:
embeddings_test = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

test = embeddings_test.embed_query("Географија е наука за Земјата.")
print(f"✅ Embeddings working, vector size: {len(test)}")

/tmp/ipykernel_7656/4283570401.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_test = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Embeddings working, vector size: 1024


Cell 12 — Check Ollama models + GPU


In [5]:
!ollama list

r = requests.get("http://localhost:11434/api/tags")
print(r.json())

NAME                                                     ID              SIZE      MODIFIED       
hf.co/NikolayKozloff/MKLLM-7B-Instruct-Q8_0-GGUF:Q8_0    547937734a0d    7.7 GB    12 minutes ago    
mkllm-7b-instruct:latest                                 154d7df49c9d    7.7 GB    12 minutes ago    
{'models': [{'name': 'hf.co/NikolayKozloff/MKLLM-7B-Instruct-Q8_0-GGUF:Q8_0', 'model': 'hf.co/NikolayKozloff/MKLLM-7B-Instruct-Q8_0-GGUF:Q8_0', 'modified_at': '2026-06-10T09:08:49.12181977Z', 'size': 7695876003, 'digest': '547937734a0d691070665e9e03b4d6fb8be2520d518d8da8b155f255d6e9ccd8', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '7.24B', 'quantization_level': 'Q8_0', 'context_length': 32768, 'embedding_length': 4096}, 'capabilities': ['completion']}, {'name': 'mkllm-7b-instruct:latest', 'model': 'mkllm-7b-instruct:latest', 'modified_at': '2026-06-10T09:08:49.195826051Z', 'size': 7695876312, 'digest': '154d7df49c9dc999a60

Cell 13 — nvidia-smi


In [6]:
!nvidia-smi

Wed Jun 10 09:21:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             26W /   70W |    2301MiB /  15360MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Cell 14 — PDF reader


In [7]:
### PDF READER
from typing import Optional
from google.colab import files
import fitz

def upload_and_read_pdf(output_path: str = "text.txt", start_page: int = 0, end_page: Optional[int] = None):
    print("Select a PDF file to upload...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded")
        return "", ""

    pdf_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {pdf_filename}")

    return pdf_filename, read_pdf_to_text(pdf_filename, output_path, start_page, end_page)

def read_pdf_to_text(pdf_path: str, output_path: str = "text.txt", start_page: int = 0, end_page: Optional[int] = None):
    try:
        text = ""
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        if end_page is None or end_page > total_pages:
            end_page = total_pages

        for page_num in range(start_page, end_page):
            page = doc.load_page(page_num)
            page_text = page.get_text()
            text += f"\n\n[PAGE {page_num + 1}]\n{page_text}"

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

        print(f"✅ Extracted {len(text)} characters from pages {start_page + 1}-{end_page} of {total_pages}")
        print(f"Saved to: {output_path}")
        return text
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""

pdf_path, extracted_text = upload_and_read_pdf(
    output_path="text.txt",
    start_page=0,
    end_page=50
)

Select a PDF file to upload...


Saving Geografija I godina gimnazija_MKD.pdf to Geografija I godina gimnazija_MKD.pdf
✅ Uploaded: Geografija I godina gimnazija_MKD.pdf
✅ Extracted 106689 characters from pages 1-50 of 128
Saved to: text.txt


Cell 15 — Config ← changed model name + output filenames


In [8]:
config = {
    "llm": {
        "model_type": "ollama",
        "model_name": "mkllm-7b-instruct",
        "temperature": 0.1,
    },
    "paths": {
        "output_dir": "./output",
        "triples_file": "mkllm7b_triples_v1.txt",
        "semantic_blocks_file": "mkllm7b_semanticBlocks_v1.txt"
    }
}

os.makedirs(config["paths"]["output_dir"], exist_ok=True)
print("Config loaded")

Config loaded


Cell 16 — initialize_llm


In [9]:
### LLM AND EMBEDDINGS
def initialize_llm(config):
    if not config:
        return None, None
    llm_config = config.get("llm", {})
    try:
        if llm_config.get("model_type") == "ollama":
            model_name = llm_config.get("model_name", "mkllm-7b-instruct")
            llm = ChatOllama(
                model=model_name,
                temperature=llm_config.get("temperature", 0.1),
                base_url="http://localhost:11434",
                format="json",
                num_predict=800,
                num_ctx=4096,
                repeat_penalty=1.3,
            )
            embeddings = HuggingFaceEmbeddings(
                model_name="intfloat/multilingual-e5-large",
                model_kwargs={"device": "cuda"},
                encode_kwargs={"normalize_embeddings": True}
            )
            print(f"✅ LLM initialized: {model_name}")
            print(f"✅ Embeddings initialized: multilingual-e5-large (cuda)")
            return llm, embeddings
        print("Only 'ollama' model type is supported")
        return None, None
    except Exception as e:
        print(f"❌ Error initializing LLM: {e}")
        return None, None

Cell 17 — Schema + prompt (unchanged)


In [16]:
### SCHEMA AND PROMPT - SIMPLIFIED

from pydantic import BaseModel, Field, field_validator
from typing import Optional, List

class SimpleTriple(BaseModel):
    subject: str = Field(default="", description="Субјект - максимум 3 збора на македонски")
    predicate: str = Field(default="", description="Предикат - еден глагол на македонски, максимум 3 збора")
    obj: str = Field(default="", description="Објект - максимум 3 збора на македонски")

class ChunkTriples(BaseModel):
    triples: List[SimpleTriple] = Field(default_factory=list, description="Листа од максимум 5 тројки")

    @field_validator("triples", mode="before")
    @classmethod
    def limit_triples(cls, v):
        if isinstance(v, list):
            return v[:5]
        return []


def get_extraction_query():
    return """
# ПРАВИЛА - СТРОГО СЛЕДИ:
- Извлечи МАКСИМУМ 5 тројки (субјект, предикат, објект) од текстот.
- СЕ МОРА да биде на МАКЕДОНСКИ ЈАЗИК.
- Субјект и Објект: максимум 3 збора, конкретен поим или ентитет.
- Предикат: еден глагол, максимум 3 збора (пр: "предизвикува", "се состои од", "се наоѓа во").
- НИКОГАШ не повторувај иста тројка.
- НИКОГАШ не измислувај факти кои не се во текстот.
- Ако нема јасни односи, врати празна листа.

# ПРИМЕРИ НА ДОБРИ ТРОЈКИ:
- субјект: "тектонски плочи", предикат: "предизвикуваат", објект: "земјотреси"
- субјект: "литосфера", предикат: "се состои од", објект: "тектонски плочи"
- субјект: "македонски јазик", предикат: "припаѓа на", објект: "словенски јазици"
- субјект: "охридско езеро", предикат: "се наоѓа во", објект: "македонија"

# НИКОГАШ НЕ ВРАЌАЈ:
- Англиски зборови
- Субјект или Објект подолги од 3 збора
- Повторени тројки
- Тројки без јасен однос во текстот
"""

17.1 Restart Ollama to not scroll up

In [17]:
import subprocess, time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(8)
print("Ollama server started")

Ollama server started


Cell 18 — Helper functions (unchanged — full code, identical to original)


In [18]:
### HELPER FUNCTIONS



# =========================
# MODELS
# =========================

nlp_coref = spacy.load("xx_ent_wiki_sm")
nlp_coref.add_pipe("sentencizer")

spacy_nlp = spacy.load("mk_core_news_lg")

# =========================
# CONSTANTS
# =========================

TRIVIAL_OBJECTS = {
    "yes", "no", "none", "unknown",
    "да", "не", "нема",
    "непознато", "непозната", "непознат"
}

MACEDONIAN_BAD_WORDS = {
    "it", "this", "that", "they", "he", "she",
    "тоа", "ова", "овој", "оваа", "тие",
    "тој", "таа", "нешто", "некој",
    "некоја", "сите", "никој", "ништо",
    "таму", "тука"
}

FORMULA_PATTERN = re.compile(r"^(?:[A-Z][a-z]?\d*)+(?:[+\-()\[\]·.]*\d*)*$")

PREDICATE_MAPPINGS = {
    "предизвикува": "causes",
    "предизвикуваат": "causes",
    "содржи": "contains",
    "содржат": "contains",
    "состои": "consists_of",
    "припаѓа": "belongs_to",
    "наоѓа": "located_in",
    "вклучува": "includes",
    "опфаќа": "includes",
    "дефинира": "defines",
    "претставува": "represents",
    "настанува": "formed_by",
    "формира": "forms",
    "тече": "flows_through",
    "граничи": "borders",
    "влијае": "affects",
    "еруптира": "erupts_from",
    "изучува": "studies"
}

# =========================
# HELPERS
# =========================

def safe_get_name(entity, default="unknown"):
    try:
        if entity is None:
            return default
        if hasattr(entity, "name") and entity.name:
            return str(entity.name)
        if isinstance(entity, str):
            return entity
        if isinstance(entity, dict) and "name" in entity:
            return str(entity["name"])
    except Exception:
        pass
    return default

def clean_entity(entity: str) -> str:
    if not entity:
        return ""

    entity = str(entity).strip()
    entity = re.sub(r"\s+", " ", entity)
    entity = re.sub(r"[,;:.!?]+$", "", entity)
    entity = entity.strip("()[]{} ")

    if not entity:
        return ""
    if entity.lower() in TRIVIAL_OBJECTS:
        return ""
    if len(entity.split()) > 8:
        return ""
    if len(entity) < 2:
        return ""
    if FORMULA_PATTERN.fullmatch(entity):
        return entity
    entity = re.sub(r"^(и|или|на|во|со)\s+", "", entity.lower()).strip()
    return entity

def clean_predicate(predicate: str) -> str:
    if not predicate:
        return ""
    predicate = str(predicate).lower().strip()
    predicate = re.sub(r"[^\w\sа-шѓќљњѕџјА-ШЃЌЉЊЅЏЈ]", "", predicate)
    predicate = re.sub(r"\s+", "_", predicate)
    if len(predicate.split("_")) > 4:
        return ""
    if len(predicate) > 40:
        return ""
    if predicate in PREDICATE_MAPPINGS:
        return PREDICATE_MAPPINGS[predicate]
    return predicate if len(predicate) >= 2 else ""

def is_valid_triple(subject: str, predicate: str, obj: str) -> bool:

    if not subject or not predicate or not obj:
        return False
    if len(subject) < 2 or len(obj) < 2:
        return False
    if len(predicate) < 2:
        return False
    if subject.lower() == obj.lower():
        return False
    if subject.lower() in MACEDONIAN_BAD_WORDS:
        return False
    if obj.lower() in MACEDONIAN_BAD_WORDS:
        return False
    if predicate in {"има", "биде", "е", "се"}:
        return False
    if re.match(r"^_+$", predicate):
        return False
    if len(subject) > 80:
        return False
    if len(obj) > 80:
        return False
    if len(predicate) > 40:
        return False
    if subject.isdigit() or obj.isdigit():
        return False
    return True

def resolve_coreferences(text: str) -> str:
    doc = spacy_nlp(text)
    resolved_sentences = []

    for sent in doc.sents:
        last_entity = None
        tokens = []

        for token in sent:
            if token.pos_ in ("PROPN", "NOUN"):
                if len(token.text) > 2 and token.text.lower() not in MACEDONIAN_BAD_WORDS:
                    subtree = " ".join(t.text for t in token.subtree)
                    cleaned = clean_entity(subtree)
                    if cleaned:
                        last_entity = cleaned

            if token.text.lower() in MACEDONIAN_BAD_WORDS and last_entity:
                tokens.append(last_entity)
            else:
                tokens.append(token.text)

        resolved_sentences.append(" ".join(tokens))

    return " ".join(resolved_sentences)

def extract_ner_entities(text: str) -> set:
    doc = spacy_nlp(text)
    entities = set()

    for ent in doc.ents:
        cleaned = clean_entity(ent.text)
        if cleaned and len(cleaned) > 2:
            entities.add(cleaned.lower())

    for token in doc:
        if token.pos_ in ("PROPN", "NOUN"):
            phrase_tokens = []
            for child in token.children:
                if child.dep_ in ("amod", "compound", "flat", "nmod"):
                    phrase_tokens.append(child.text)
            phrase_tokens.append(token.text)
            phrase = " ".join(phrase_tokens)
            cleaned = clean_entity(phrase)
            if cleaned and len(cleaned) > 2:
                entities.add(cleaned.lower())

    return entities

def is_entity_known(entity: str, known_entities: set) -> bool:
    entity_lower = entity.lower()

    if entity_lower in known_entities:
        return True

    for word in entity_lower.split():
        if word in known_entities and len(word) > 3:
            return True

    return False

def extract_svo_from_blocks(blocks: List[str]) -> List[tuple]:
    triples = []

    for block in blocks:
        text = re.sub(r"^[^:]+:\s*", "", block).strip()

        if len(text) < 10:
            continue

        try:
            doc = spacy_nlp(text)

            for sent in doc.sents:

                for token in sent:

                    if token.pos_ != "VERB":
                        continue

                    subjects = []

                    for child in token.children:
                        if child.dep_ in ("nsubj", "nsubj:pass"):
                            subj_phrase = " ".join(t.text for t in child.subtree)
                            subj_phrase = clean_entity(subj_phrase)

                            if subj_phrase:
                                subjects.append(subj_phrase)

                    objects = []

                    for child in token.children:

                        if child.dep_ in ("obj", "iobj", "obl"):

                            obj_phrase = " ".join(t.text for t in child.subtree)
                            obj_phrase = clean_entity(obj_phrase)

                            if obj_phrase:
                                objects.append(obj_phrase)

                        if child.dep_ == "conj":

                            conj_phrase = " ".join(t.text for t in child.subtree)
                            conj_phrase = clean_entity(conj_phrase)

                            if conj_phrase:
                                objects.append(conj_phrase)

                    predicate = clean_predicate(token.lemma_)

                    for s in subjects:
                        for o in objects:

                            if is_valid_triple(s, predicate, o):
                                triples.append((s, predicate, o))

        except Exception:
            continue

    return triples

def deduplicate_triples(triples: List[tuple]) -> List[tuple]:
    seen = set()
    unique = []

    for s, p, o in triples:
        key = (s.lower(), p.lower(), o.lower())

        if key not in seen:
            seen.add(key)
            unique.append((s, p, o))

    return unique

def save_triples(triples, filename="extracted_triples.txt"):
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Knowledge Graph Triples\n")
        f.write(f"# Total: {len(triples)} triples\n")
        f.write("# Format: (Subject) --[Predicate]--> (Object)\n\n")

        for i, (s, p, o) in enumerate(triples, 1):
            f.write(f"{i:3d}. ({s}) --[{p}]--> ({o})\n")

        f.write(f"\n# Extraction completed. {len(triples)} unique triples saved.\n")

class PipelineError(Exception):
    pass

async def robust_distillation(chunk: str, distiller, max_retries: int = 3):
    for attempt in range(max_retries):
        try:
            cleaned_chunk = chunk.replace("{", "[").replace("}", "]")

            return await distiller.distill(
                documents=[cleaned_chunk],
                IE_query=get_extraction_query(),
                output_data_structure=ChunkTriples
            )

        except asyncio.TimeoutError as e:
            print(f"    -> Attempt {attempt + 1} timed out: {e}")

        except Exception as e:
            error_type = type(e).__name__
            error_msg = str(e)[:100] + "..." if len(str(e)) > 100 else str(e)

            print(f"    -> Attempt {attempt + 1} failed ({error_type}): {error_msg}")

            if attempt == max_retries - 1:
                raise PipelineError(
                    f"Distillation failed after {max_retries} attempts: "
                    f"{error_type} - {error_msg}"
                )

        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"    -> Retrying in {wait_time} seconds...")
            await asyncio.sleep(wait_time)

def create_semantic_blocks_from_triples(chunk_triples: ChunkTriples) -> List[str]:
    blocks = []

    if not chunk_triples or not chunk_triples.triples:
        return blocks

    for triple in chunk_triples.triples:
        s = triple.subject.strip()
        p = triple.predicate.strip()
        o = triple.obj.strip()

        if s and p and o:
            blocks.append(f"{s} {p} {o}")

    return blocks

def process_chunks_in_batches(chunks, batch_size=5):
    for i in range(0, len(chunks), batch_size):
        yield chunks[i:i + batch_size]

class ETACalculator:
    def __init__(self, total_items: int):
        self.total_items = max(total_items, 1)
        self.start_time = time.time()

    def get_eta_string(self, current_item: int) -> str:
        elapsed = time.time() - self.start_time
        avg = elapsed / current_item if current_item else 0
        remaining = avg * (self.total_items - current_item)
        progress = (current_item / self.total_items) * 100

        return (
            f"Progress: {progress:.1f}% | "
            f"Elapsed: {elapsed/60:.1f}min | "
            f"ETA: {remaining/60:.1f}min"
        )

Cell 19 — Main pipeline (unchanged)


In [19]:
### MAIN PIPELINE

async def main(input_source: str = "text.txt"):
    print("Starting Macedonian Textbook KG Extraction Pipeline\n")

    if os.path.exists(input_source):
        with open(input_source, "r", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        text = input_source.strip()

    if len(text) < 20:
        print("Text too short")
        return []

    llm, embeddings = initialize_llm(config)

    if llm is None or embeddings is None:
        print("LLM or embeddings failed to initialize")
        return []

    print(f"✅ LLM initialized: {config['llm']['model_name']}")

    output_dir = config.get("paths", {}).get("output_dir", "./output")
    triples_file = config.get("paths", {}).get("triples_file", "textbook_triples_v1.txt")
    semantic_blocks_file = config.get("paths", {}).get("semantic_blocks_file", "textbook_semanticBlocks_v1.txt")

    os.makedirs(output_dir, exist_ok=True)

    all_triples = []
    nlp_triples_count = 0
    distiller_triples_count = 0
    star_triples_count = 0

    start_time = time.time()

    # =========================================================
    # STEP 0 - COREFERENCE
    # =========================================================

    print("\n=== STEP 0: Coreference Resolution ===")

    try:
        resolved_text = resolve_coreferences(text)
        print(f"✅ Done ({len(text)} → {len(resolved_text)} chars)")
    except Exception as e:
        print(f"⚠️ Coreference skipped: {e}")
        resolved_text = text

    # =========================================================
    # STEP 0.5 - NER
    # =========================================================

    print("\n=== STEP 0.5: NER Entity Extraction ===")

    try:
        known_entities = extract_ner_entities(resolved_text)
        print(f"✅ Extracted {len(known_entities)} known entities")
    except Exception as e:
        print(f"⚠️ NER skipped: {e}")
        known_entities = set()

    # =========================================================
    # STEP 1 - CHUNKING
    # =========================================================

    print("\n=== STEP 1: Chunking ===")

    sentences = re.split(r"(?<=[.!?؟])\s+|\n+", resolved_text)

    CHUNK_SIZE = 800
    OVERLAP = 150

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        if len(current_chunk + sentence) <= CHUNK_SIZE:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())

            overlap_text = (
                current_chunk[-OVERLAP:]
                if len(current_chunk) > OVERLAP
                else current_chunk
            )

            current_chunk = overlap_text + sentence + " "

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(
        f"✅ Text chunked: {len(resolved_text)} chars "
        f"in {len(chunks)} chunks "
        f"(size={CHUNK_SIZE}, overlap={OVERLAP})"
    )

    # =========================================================
    # STEP 2 - DISTILLER
    # =========================================================

    print("\n=== STEP 2: Direct Triple Extraction via Distiller ===")

    eta_distiller = ETACalculator(len(chunks))
    all_semantic_blocks = []

    try:
        from itext2kg.llm_output_parsing.langchain_output_parser import (
            LangchainOutputParser,
            ProviderType,
            PROVIDER_CONFIGS,
            ProviderConfig
        )

        PROVIDER_CONFIGS[ProviderType.UNKNOWN] = ProviderConfig(
            name="Ollama",
            max_elements_per_batch=1,
            max_tokens_per_batch=4000,
            max_context_window=131072,
            max_pending_requests=None,
            sleep_between_batches=0.5,
        )

        original_detect = LangchainOutputParser._detect_provider

        def patched_detect(self):
            cls = self.model.__class__.__name__.lower()
            mod = self.model.__class__.__module__.lower()

            if "ollama" in cls or "ollama" in mod:
                return ProviderType.UNKNOWN

            return original_detect(self)

        LangchainOutputParser._detect_provider = patched_detect

        print("✅ Ollama provider patch applied")

        distiller = DocumentsDistiller(llm_model=llm)

        import types

        async def patched_extract(
            self,
            output_data_structure,
            contexts,
            system_query=None
        ):
            if system_query is None:
                system_query = "Act like an experienced information extractor."

            structured_llm = self.model.with_structured_output(
                output_data_structure
            )

            outputs = []

            for i, context in enumerate(contexts):

                prompt = (
                    f"# Context: {context}\n\n"
                    f"# Question: {system_query}\n\n"
                    f"Answer: "
                )

                for attempt in range(3):

                    try:
                        result = await structured_llm.ainvoke(prompt)

                        outputs.append(result)

                        print(
                            f"    ✅ Context {i+1}/{len(contexts)} extracted"
                        )

                        break

                    except Exception as e:

                        if attempt == 2:
                            print(
                                f"    ❌ Context {i+1} failed: "
                                f"{str(e)[:80]}"
                            )

                            outputs.append(output_data_structure())

                        else:
                            print(
                                f"    ⚠️ Attempt {attempt+1} failed: "
                                f"{str(e)[:60]}"
                            )

                            await asyncio.sleep(2 ** attempt)

                await asyncio.sleep(0.5)

            return outputs

        parser = distiller.langchain_output_parser

        parser.extract_information_as_json_for_context = (
            types.MethodType(patched_extract, parser)
        )

        print("✅ Distiller instance patched")

    except ImportError as e:
        print(f"❌ Failed to import DocumentsDistiller: {e}")
        return []

    successful_chunks = 0
    BATCH_SIZE = 5

    for batch_num, chunk_batch in enumerate(
        process_chunks_in_batches(chunks, BATCH_SIZE)
    ):

        print(
            f"\nBATCH {batch_num + 1}/"
            f"{(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}"
        )

        for i, chunk in enumerate(chunk_batch):

            global_chunk_idx = batch_num * BATCH_SIZE + i

            print(f"Chunk {global_chunk_idx + 1}/{len(chunks)}...")

            try:
                result = await robust_distillation(chunk, distiller)

                if result and hasattr(result, "triples") and result.triples:

                    for triple in result.triples:

                        s = clean_entity(triple.subject)
                        p = clean_predicate(triple.predicate)
                        o = clean_entity(triple.obj)

                        if is_valid_triple(s, p, o):
                            all_triples.append((s, p, o))
                            distiller_triples_count += 1

                    blocks = create_semantic_blocks_from_triples(result)

                    all_semantic_blocks.extend(blocks)

                    successful_chunks += 1

                    print(
                        f"  → {len(result.triples)} triples extracted, "
                        f"{len(blocks)} blocks created"
                    )

                else:
                    all_semantic_blocks.append(chunk[:400])

            except PipelineError as e:

                print(
                    f"  ⚠️ Chunk failed, using raw fallback: "
                    f"{str(e)[:80]}"
                )

                all_semantic_blocks.append(chunk[:400])

            print(
                eta_distiller.get_eta_string(global_chunk_idx + 1)
            )

        gc.collect()

    print(f"\n✅ Distillation done: {successful_chunks}/{len(chunks)} chunks")
    print(f"✅ Direct triples from distiller: {distiller_triples_count}")
    print(f"✅ Semantic blocks: {len(all_semantic_blocks)}")

    # =========================================================
    # STEP 2.5 - NLP SVO EXTRACTION
    # =========================================================

    print("\n=== STEP 2.5: NLP SVO Extraction ===")

    try:
        nlp_triples = extract_svo_from_blocks(all_semantic_blocks)

        nlp_triples_count = len(nlp_triples)

        all_triples.extend(nlp_triples)

        print(f"✅ NLP extracted {nlp_triples_count} triples")

        for s, p, o in nlp_triples[:5]:
            print(f"   ({s}) --[{p}]--> ({o})")

    except Exception as e:
        print(f"⚠️ NLP extraction failed: {e}")

    # =========================================================
    # STEP 3 - STAR
    # =========================================================
    # print("\n=== STEP 3: STAR Knowledge Graph Extraction ===")
    # seen_blocks = set()
    # unique_blocks = []
    # for b in all_semantic_blocks:
    #     key = b.lower().strip()
    #     if key not in seen_blocks and len(key) > 10:
    #         seen_blocks.add(key)
    #         unique_blocks.append(b)
    # all_semantic_blocks = unique_blocks
    # print(f"Unique blocks for STAR: {len(all_semantic_blocks)}")
    # try:
    #     star_model = iText2KG_Star(
    #         llm_model=llm,
    #         embeddings_model=embeddings
    #     )
    #     star_parser = star_model.langchain_output_parser
    #     star_parser.extract_information_as_json_for_context = (
    #         types.MethodType(patched_extract, star_parser)
    #     )
    #     print("✅ STAR model patched")
    #     batches = [
    #         all_semantic_blocks[i:i+10]
    #         for i in range(0, len(all_semantic_blocks), 10)
    #     ]
    #     eta_star = ETACalculator(len(batches))
    #     for batch_idx, batch in enumerate(batches):
    #         print(f"\nSTAR BATCH {batch_idx + 1}/{len(batches)}")
    #         for block_idx, block in enumerate(batch):
    #             try:
    #                 kg_result = await star_model.build_graph(
    #                     sections=[block],
    #                     ent_threshold=0.5,
    #                     rel_threshold=0.4,
    #                     observation_date="2026-06-08"
    #                 )
    #                 if (
    #                     hasattr(kg_result, "relationships")
    #                     and kg_result.relationships
    #                 ):
    #                     block_triples = 0
    #                     for relationship in kg_result.relationships:
    #                         s = clean_entity(
    #                             safe_get_name(
    #                                 getattr(
    #                                     relationship,
    #                                     "startEntity",
    #                                     None
    #                                 )
    #                             )
    #                         )
    #                         o = clean_entity(
    #                             safe_get_name(
    #                                 getattr(
    #                                     relationship,
    #                                     "endEntity",
    #                                     None
    #                                 )
    #                             )
    #                         )
    #                         p = clean_predicate(
    #                             safe_get_name(
    #                                 getattr(
    #                                     relationship,
    #                                     "name",
    #                                     None
    #                                 ),
    #                                 ""
    #                             )
    #                         )
    #                         if is_valid_triple(s, p, o):
    #                             if (
    #                                 not known_entities
    #                                 or is_entity_known(s, known_entities)
    #                                 or is_entity_known(o, known_entities)
    #                             ):
    #                                 all_triples.append((s, p, o))
    #                                 block_triples += 1
    #                                 star_triples_count += 1
    #                     print(
    #                         f"  Block {block_idx + 1}: "
    #                         f"{block_triples} triples"
    #                     )
    #             except Exception as block_error:
    #                 print(
    #                     f"  Block {block_idx + 1} failed: "
    #                     f"{str(block_error)[:80]}"
    #                 )
    #         print(eta_star.get_eta_string(batch_idx + 1))
    # except Exception as star_error:
    #     print(f"STAR failed: {str(star_error)[:200]}")

    # =========================================================
    # STEP 4 - POST PROCESSING
    # =========================================================

    print("\n=== STEP 4: Post-Processing ===")

    final_triples = deduplicate_triples([
        (
            clean_entity(s),
            clean_predicate(p),
            clean_entity(o)
        )
        for s, p, o in all_triples
        if is_valid_triple(
            clean_entity(s),
            clean_predicate(p),
            clean_entity(o)
        )
    ])

    elapsed_minutes = (time.time() - start_time) / 60

    print(f"\n✅ Processing completed in {elapsed_minutes:.1f} minutes")
    print(f"Distiller triples: {distiller_triples_count}")
    print(f"NLP triples: {nlp_triples_count}")
    # print(f"STAR triples: {star_triples_count}")
    print(f"Final unique valid triples: {len(final_triples)}")

    triples_path = os.path.join(output_dir, triples_file)
    blocks_path = os.path.join(output_dir, semantic_blocks_file)

    save_triples(final_triples, triples_path)

    print(f"Triples saved to: {triples_path}")

    with open(blocks_path, "w", encoding="utf-8") as f:

        f.write(
            f"# Semantic Blocks\n"
            f"# Total: {len(all_semantic_blocks)}\n\n"
        )

        for i, block in enumerate(all_semantic_blocks, 1):

            f.write(
                f"Block {i}:\n"
                f"{block}\n"
                f"{'-'*50}\n\n"
            )

    print(f"Blocks saved to: {blocks_path}")

    if final_triples:

        print("\nSample extracted triples:")

        for i, (s, p, o) in enumerate(final_triples[:10], 1):
            print(f"   {i:2d}. ({s}) --[{p}]--> ({o})")

    return final_triples

Cell 20 — Run pipeline (unchanged)


In [20]:
### RUN PIPELINE

import nest_asyncio
import asyncio
nest_asyncio.apply()

async def run_textbook_pipeline():
    triples = await main("text.txt")
    print(f"\nPipeline completed! Found {len(triples)} triples")
    return triples

triples = await run_textbook_pipeline()

Starting Macedonian Textbook KG Extraction Pipeline



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ LLM initialized: mkllm-7b-instruct
✅ Embeddings initialized: multilingual-e5-large (cuda)
✅ LLM initialized: mkllm-7b-instruct

=== STEP 0: Coreference Resolution ===
✅ Done (106686 → 114127 chars)

=== STEP 0.5: NER Entity Extraction ===
✅ Extracted 1997 known entities

=== STEP 1: Chunking ===
✅ Text chunked: 114127 chars in 173 chunks (size=800, overlap=150)

=== STEP 2: Direct Triple Extraction via Distiller ===
✅ Ollama provider patch applied
[2026-06-10 09:45:46] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 🔍 Detected LLM Provider: Ollama
[2026-06-10 09:45:46] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 📊 Rate Limiting Config: 1 requests/batch, 4000 tokens/batch
✅ Distiller instance patched

BATCH 1/35
Chunk 1/173...
    ✅ Context 1/1 extracted
  → 2 triples extracted, 2 blocks created
Progress: 0.6% | Elapsed: 0.8min | ETA: 140.8min
Chunk 2/173...
    ✅ Context 1/1 extracted
  → 2 triples extracted, 2 blocks created
Progress: 1.2% | Ela

Cell 21 — Download outputs ← changed filenames


In [21]:
from google.colab import files
files.download("./output/mkllm7b_triples_v1.txt")
files.download("./output/mkllm7b_semanticBlocks_v1.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>